In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Playwright 패키지 설치
!pip install playwright

!playwright install chromium
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 12.2 MB/s eta 0:00:00
(node:19252) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [] 0% 0.0s164.7 MiB [] 0% 211.6s164.7 MiB [] 0% 221.7s164.7 MiB [] 0% 696.4s164.7 MiB [] 0% 582.8s164.7 MiB [] 0% 795.1s164.7 MiB [] 0% 702.8s164.7 MiB [] 0% 636.9s164.7 MiB [] 0% 586.2s164.7 MiB [] 0% 547.9s164.7 MiB [] 0% 537.3s164.7 MiB [] 0% 510.4s164.7 MiB [] 0% 487.1s164.7 MiB [] 0% 464.2s164.7 MiB [] 0% 448.3s164.7 MiB [] 0% 432.4s164.7 MiB [] 0% 417.2s164.7 MiB [] 0% 402.5s164.7 MiB [] 0% 389.4s164.7 MiB [] 0% 377.8s164.7 MiB [] 0% 351.2s164.7 MiB [] 0% 328.9s164.7 MiB [] 0% 310.9s164.7 MiB [] 0% 294.8s164.7 MiB [] 0% 280.6s164.7 MiB [] 0% 268.5s164.7 MiB [] 0% 257.6s164.7 MiB [] 0% 249.2s164.7 

In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import json
import os
from datetime import datetime, timedelta
from tqdm.asyncio import tqdm

# 0. 설정 및 유틸리티
SAVE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data"
START_DATE = "2026-01-14"
END_DATE = "2026-01-26" # 기간 설정
BASE_URL = "https://game.naver.com/esports/League_of_Legends/news/lol"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR, exist_ok=True)

def generate_date_range(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(delta.days + 1)]

# 1. [함수] 링크 수집
async def step1_get_links():
    dates = generate_date_range(START_DATE, END_DATE)
    collected_links = []

    print(f"[링크 수집 시작]: ({START_DATE} ~ {END_DATE})")

    async with async_playwright() as p:
        # 브라우저 실행
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-setuid-sandbox"])
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )
        page = await context.new_page()

        for date in tqdm(dates, desc="목록 스캔 중"):
            target_url = f"{BASE_URL}?date={date}"
            try:
                await page.goto(target_url, timeout=30000, wait_until="domcontentloaded")

                # 목록 로딩 대기
                try:
                    await page.wait_for_selector("li[class*='news_card_item']", timeout=6000)
                except:
                    continue

                # 스크롤
                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                await asyncio.sleep(1)

                # 링크 추출
                cards = await page.query_selector_all("li[class*='news_card_item']")
                for card in cards:
                    link_el = await card.query_selector("a")
                    if link_el:
                        href = await link_el.get_attribute("href")
                        # 제목 미리보기 (본문 못 찾을 때 대비용)
                        title_el = await card.query_selector("strong[class*='news_card_title']")
                        title_preview = await title_el.inner_text() if title_el else "No Title"

                        if href:
                            collected_links.append({
                                "date": date,
                                "url": href,
                                "title": title_preview
                            })
            except Exception as e:
                print(e)
                continue

        await browser.close()

    # 중간 결과 저장
    save_path = os.path.join(SAVE_DIR, "step1_links.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(collected_links, f, ensure_ascii=False, indent=4)

    print(f"[링크 수집 완료] 총 {len(collected_links)}개 링크 저장.")
    print(f"중간 파일 저장됨: {save_path}")

    return collected_links

# 2. [함수] 본문 크롤링
async def step2_get_contents(link_list):
    if not link_list:
        return []

    print(f"[본문 크롤링 시작] (대상: {len(link_list)}개)")
    final_dataset = []

    async with async_playwright() as p:
        # 브라우저 실행
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-setuid-sandbox"])
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        for item in tqdm(link_list, desc="본문 수집 중"):
            url = item["url"]

            try:
                # 페이지 접속
                await page.goto(url, timeout=15000, wait_until="domcontentloaded")
                await asyncio.sleep(0.5) # 로딩 대기

                # HTML을 통째로 가져와서 파이썬(BS4)이 찾음
                html = await page.content()
                soup = BeautifulSoup(html, "html.parser")

                # 1. 본문 찾기
                content_candidates = [
                    soup.select_one("#newsct_article"),          # 최신 표준
                    soup.select_one(".NewsEndMain_article_body"), # PC e스포츠 전용
                    soup.select_one("#dic_area"),                # 구형
                    soup.select_one("._article_body")            # 범용 클래스
                ]

                content_elem = None
                for candidate in content_candidates:
                    if candidate:
                        content_elem = candidate
                        break

                # 2. 제목 찾기
                title_candidates = [
                    soup.select_one(".media_end_head_headline"),
                    soup.select_one("#title_area"),
                    soup.select_one(".NewsEndMain_title")
                ]

                title_text = item["title"] # 기본값은 목록에서 가져온 제목
                for t in title_candidates:
                    if t:
                        title_text = t.get_text(strip=True)
                        break

                if content_elem:
                    # 불필요한 태그(스크립트, 광고, 기자정보) 제거
                    for tag in content_elem.select("script, style, iframe, .img_desc, .end_photo_org, .reporter_area"):
                        tag.decompose()

                    body_text = content_elem.get_text(separator=" ", strip=True)

                    if len(body_text) > 30:
                        final_dataset.append({
                            "date": item["date"],
                            "title": title_text,
                            "content": body_text,
                            "url": url
                        })

            except Exception as e:
                print(e)
                continue

        # print(final_dataset)
        await browser.close()

    return final_dataset

# 3. 메인 실행
async def main():
    # 1. 링크 수집 실행
    links = await step1_get_links()
    print(links)

    # 링크 수집 결과 확인
    if not links:
        return

    # 2. 본문 크롤링 실행
    final_data = await step2_get_contents(links)

    # 3. 최종 저장
    if final_data:
        json_path = os.path.join(SAVE_DIR, "lck_final_complete.json")
        txt_path = os.path.join(SAVE_DIR, "lck_final_complete.txt")

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(final_data, f, ensure_ascii=False, indent=4)

        with open(txt_path, "w", encoding="utf-8") as f:
            for article in final_data:
                f.write(f"날짜: {article["date"]}\n")
                f.write(f"제목: {article["title"]}\n")
                f.write(f"내용: {article["content"]}\n")
                f.write(f"링크: {article["url"]}\n")
                f.write("-" * 50 + "\n")

        print(f"\n모든 작업 완료! 총 {len(final_data)}건 저장됨.")
        print(f"파일 위치: {txt_path}")
    else:
        return

# 실행
await main()

[링크 수집 시작]: (2026-01-14 ~ 2026-01-26)


목록 스캔 중: 100%|██████████| 13/13 [00:57<00:00,  4.40s/it]


[링크 수집 완료] 총 247개 링크 저장.
중간 파일 저장됨: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/step1_links.json
[{'date': '2026-01-14', 'url': 'https://m.sports.naver.com/esports/article/450/0000148361?sid3=79b', 'title': "2026 LCK컵, '체급이 다르다' 증명한 DK... 브리온 54분 셧아웃, 장로 그룹 2연승 질주"}, {'date': '2026-01-14', 'url': 'https://m.sports.naver.com/esports/article/347/0000191232?sid3=79b', 'title': '[LCK컵] DK 김대호 감독, "누굴 집어서 칭찬하기 어려울 정도"'}, {'date': '2026-01-14', 'url': 'https://m.sports.naver.com/esports/article/005/0001826216?sid3=79b', 'title': '김대호 “알아서 잘하는 바텀 듀오, 에너지가 넘친다”'}, {'date': '2026-01-14', 'url': 'https://m.sports.naver.com/esports/article/311/0001963803?sid3=79b', 'title': '"코치 보이스는 신중하게"… 김대호 감독이 밝힌 디플러스 기아의 선택 [LCK컵] (인터뷰)'}, {'date': '2026-01-14', 'url': 'https://m.sports.naver.com/esports/article/109/0005462415?sid3=79b', 'title': '[LCK컵] DK, 54분만에 브리온 2-0 완파…장로 그룹 2연승'}, {'date': '2026-01-14', 'url': 'https://m.sports.naver.com/esports/article/005/0001826215?sid3=79b', 'ti

본문 수집 중: 100%|██████████| 247/247 [04:56<00:00,  1.20s/it]



모든 작업 완료! 총 247건 저장됨.
파일 위치: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/lck_final_complete.txt


In [12]:
import asyncio
from playwright.async_api import async_playwright
import nest_asyncio
import json
import os

nest_asyncio.apply()

async def crawl_lck_final():
    results = []
    target_months = ['2026-01']

    print("LCK 경기 데이터 크롤링 시작")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(viewport={"width": 1920, "height": 1080})
        page = await context.new_page()

        for month in target_months:
            url = f"https://game.naver.com/esports/League_of_Legends/schedule/lck?date={month}"
            print(f"   Now Scanning: {url}")

            try:
                # 1. 접속 및 로딩 대기
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)

                # 데이터가 로딩될 때까지 대기
                try:
                    await page.wait_for_selector("div[class*='card_item']", state='attached', timeout=15000)
                except:
                    continue

                # 2. 카드 순회
                cards = await page.locator("div[class*='card_item']").all()

                for card in cards:
                    # 날짜 추출
                    date_el = card.locator("div[class*='card_date']")
                    if await date_el.count() > 0:
                        raw_date = await date_el.inner_text()
                        match_date = raw_date.replace("\n", " ").strip()
                    else:
                        match_date = "날짜 미상"

                    # 3. 경기(row) 리스트 순회
                    rows = await card.locator("li[class*='row_item']").all()

                    for row in rows:
                        try:
                            # (1) 시간 및 상태 추출
                            time_el = row.locator("span[class*='row_time']")
                            state_el = row.locator("span[class*='row_state']")

                            match_time = await time_el.inner_text() if await time_el.count() > 0 else "시간미상"
                            match_state = await state_el.inner_text() if await state_el.count() > 0 else "상태미상"

                            # (2) 홈 팀 정보 추출 (row_home)
                            home_div = row.locator("div[class*='row_home']")
                            if await home_div.count() == 0: continue

                            team_a = await home_div.locator("span[class*='row_name']").inner_text()
                            score_a_el = home_div.locator("span[class*='row_score']")
                            score_a = await score_a_el.inner_text() if await score_a_el.count() > 0 else "0"

                            # (3) 어웨이 팀 정보 추출 (row_away)
                            away_div = row.locator("div[class*='row_away']")

                            team_b = await away_div.locator("span[class*='row_name']").inner_text()
                            score_b_el = away_div.locator("span[class*='row_score']")
                            score_b = await score_b_el.inner_text() if await score_b_el.count() > 0 else "0"

                            # (4) 데이터 정제 및 저장
                            full_score = f"{score_a}:{score_b}"

                            match_info = {
                                "date": match_date,
                                "time": match_time,
                                "team_a": team_a,
                                "team_b": team_b,
                                "score": full_score,
                                "status": match_state,
                                "description": f"{match_date}에 진행된 {team_a} 대 {team_b}의 경기는 {score_a}:{score_b} 스코어로 {match_state}되었습니다."
                            }

                            results.append(match_info)

                        except Exception as e:
                            continue

            except Exception as e:
                continue

        await browser.close()

    print(f"총 {len(results)}개의 경기 데이터를 확보했습니다.")
    return results

# 실행 및 파일 저장 로직
if __name__ == "__main__":
    # 1. 크롤링 실행
    lck_data = asyncio.run(crawl_lck_final())

    # 2. 결과 출력
    print("\n[수집 결과 확인]")
    for item in lck_data[:3]:
        print(item)

    # 3. JSON 파일 저장
    file_path = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/match_results.json"
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(lck_data, f, ensure_ascii=False, indent=4)
        print(f"\n저장 완료: {os.path.abspath(file_path)}")

    except Exception as e:
        print(e)

LCK 경기 데이터 크롤링 시작
   Now Scanning: https://game.naver.com/esports/League_of_Legends/schedule/lck?date=2026-01
총 24개의 경기 데이터를 확보했습니다.

[수집 결과 확인]
{'date': '01월 14일 (수)', 'time': '08:00', 'team_a': 'DN 수퍼스', 'team_b': 'kt 롤스터', 'score': '1:2', 'status': '종료', 'description': '01월 14일 (수)에 진행된 DN 수퍼스 대 kt 롤스터의 경기는 1:2 스코어로 종료되었습니다.'}
{'date': '01월 14일 (수)', 'time': '10:00', 'team_a': '브리온', 'team_b': 'Dplus KIA', 'score': '0:2', 'status': '종료', 'description': '01월 14일 (수)에 진행된 브리온 대 Dplus KIA의 경기는 0:2 스코어로 종료되었습니다.'}
{'date': '01월 15일 (목)', 'time': '08:00', 'team_a': '젠지', 'team_b': 'DRX', 'score': '2:0', 'status': '종료', 'description': '01월 15일 (목)에 진행된 젠지 대 DRX의 경기는 2:0 스코어로 종료되었습니다.'}

저장 완료: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/match_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>